# 02 — Athena External Table (Curated Parquet)

This notebook creates an **Athena external table** over the curated Parquet written by Notebook 01.

Why this exists:
- It supports the **Data Engineering** section of the Design Document.
- It gives you SQL access to curated buoy data (counts, sanity queries, etc.).

If you don't need Athena for your workflow, you can skip this notebook.


In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# Load from previous notebook
%store -r bucket
%store -r region
%store -r S3_PREFIX_PARQUET
%store -r S3_PREFIX_CSV
%store -r BUOY_IDS

print("Bucket:", bucket)
print("Region:", region)
print("S3_PREFIX_PARQUET:", S3_PREFIX_PARQUET)
print("S3_PREFIX_CSV:", S3_PREFIX_CSV)
print("BUOY_IDS:", BUOY_IDS)

Bucket: sagemaker-us-east-1-115800714036
Region: us-east-1
S3_PREFIX_PARQUET: curated/ndbc_parquet
S3_PREFIX_CSV: curated/ndbc_csv
BUOY_IDS: ['46086', '46042', '46011']


In [4]:
# Athena staging dir (query results)
s3_staging_dir = f"s3://{bucket}/athena/staging/"
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)
print("Athena staging dir:", s3_staging_dir)

Athena staging dir: s3://sagemaker-us-east-1-115800714036/athena/staging/


In [5]:
database_name = "ndbc_data"
table_name = "curated_stdmet"

# Parquet root (PARTITIONED BY (buoy string) expects folders buoy=XXXX/)
s3_parquet_root = f"s3://{bucket}/{S3_PREFIX_PARQUET}/"
print("Parquet root:", s3_parquet_root)

Parquet root: s3://sagemaker-us-east-1-115800714036/curated/ndbc_parquet/


In [6]:
SHOW TABLES IN ndbc_data;


╭──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ SHOW TABLES IN ndbc_data;                                                                        │
│      ▲                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
SyntaxError: invalid syntax

In [7]:
# Create database
with conn.cursor() as cur:
    cur.execute(f"CREATE DATABASE {database_name}")
print("Created database:", database_name)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 # Create database                                                                            │
│   2 with conn.cursor() as cur:                                                                   │
│ ❱ 3 │   cur.execute(f"CREATE DATABASE {database_name}")                                          │
│   4 print("Created database:", database_name)                                                    │
│   5                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pyathena/cursor.py:219 in execute                        │
│                                                                                                  │
│   216 │   │   │   │   self._retry_config,                                                        │
│   217 │   │   │   )                                                                              │
│   218 │   │   else:                                                                              │
│ ❱ 219 │   │   │   raise OperationalError(query_execution.state_change_reason)                    │
│   220 │   │   return self                                                                        │
│   221 │                                                                                          │
│   222 │   def executemany(                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
OperationalError: FAILED: Execution Error, return code 1 from org.apache.hadoop.hive.ql.exec.DDLTask. Database 
ndbc_data already exists

In [16]:
# Create external table (partitioned by buoy)
create_table_sql = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name} (
    `timestamp` timestamp,
    station_id string,
    wind_direction double,
    wind_speed double,
    wind_gust double,
    wave_height double,
    dominant_wave_period double,
    average_wave_period double,
    mean_wave_direction double,
    pressure double,
    air_temperature double,
    water_temperature double,
    dewpoint_temperature double,
    wind_speed_ms double,
    wave_energy double
)
PARTITIONED BY (`buoy` string)
STORED AS PARQUET
LOCATION '{s3_parquet_root}'
"""

print(create_table_sql)

with conn.cursor() as cur:
    cur.execute(create_table_sql)

print("Created table (if not exists):", f"{database_name}.{table_name}")


CREATE EXTERNAL TABLE IF NOT EXISTS ndbc_data.curated_stdmet (
    `timestamp` timestamp,
    station_id string,
    wind_direction double,
    wind_speed double,
    wind_gust double,
    wave_height double,
    dominant_wave_period double,
    average_wave_period double,
    mean_wave_direction double,
    pressure double,
    air_temperature double,
    water_temperature double,
    dewpoint_temperature double,
    wind_speed_ms double,
    wave_energy double
)
PARTITIONED BY (`buoy` string)
STORED AS PARQUET
LOCATION 's3://sagemaker-us-east-1-115800714036/curated/ndbc_parquet/'

Created table (if not exists): ndbc_data.curated_stdmet


In [17]:
# Load partitions (buoy=XXXX folders)
repair_sql = f"MSCK REPAIR TABLE {database_name}.{table_name}"
print(repair_sql)

with conn.cursor() as cur:
    cur.execute(repair_sql)

print("Repaired partitions.")

MSCK REPAIR TABLE ndbc_data.curated_stdmet
Repaired partitions.


In [9]:
q = f"SHOW CREATE TABLE {database_name}.{table_name}"
df_show = pd.read_sql(q, conn)
print(df_show.iloc[0,0])
q = f"SHOW PARTITIONS {database_name}.{table_name}"
pd.read_sql(q, conn)


/tmp/ipykernel_343/2975409429.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_show = pd.read_sql(q, conn)


CREATE EXTERNAL TABLE `ndbc_data.curated_stdmet`(


/tmp/ipykernel_343/2975409429.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(q, conn)


,partition
0,buoy=46086
1,buoy=46042
2,buoy=46011


In [18]:
q = f"SELECT COUNT(*) AS n FROM {database_name}.{table_name}"
pd.read_sql(q, conn)


/tmp/ipykernel_343/326860364.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(q, conn)


,n
0,0


In [11]:
# Sanity query: row counts per buoy
q = f"""
SELECT buoy, COUNT(*) AS n
FROM {database_name}.{table_name}
GROUP BY buoy
ORDER BY n DESC
"""
df_counts = pd.read_sql(q, conn)
df_counts

/tmp/ipykernel_343/424462232.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_counts = pd.read_sql(q, conn)


,buoy,n


In [10]:
# Sample data
q = f"""SELECT * FROM {database_name}.{table_name} LIMIT 10"""
pd.read_sql(q, conn)

/tmp/ipykernel_1658/278100926.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(q, conn)


,timestamp,station_id,wind_direction,wind_speed,wind_gust,wave_height,dominant_wave_period,average_wave_period,mean_wave_direction,pressure,air_temperature,water_temperature,dewpoint_temperature,wind_speed_ms,wave_energy,buoy


In [11]:
%store database_name
%store table_name

Stored 'database_name' (str)
Stored 'table_name' (str)
